# Prompting-Based Classification Model

We implement a prompting-based model for customer tweet classification.

Instead of training a model, we use a fixed zero-shot prompt that asks an LLM to classify each tweet into one of the valid company labels.

In [2]:
# Import basic libraries
import pandas as pd
import numpy as np

# Import evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [3]:
# Load test data created in Notebook 2
test_df = pd.read_csv("../Data/test_tweets.csv")

# Show dataset shape and preview
print("Test dataset shape:", test_df.shape)
test_df.head()

Test dataset shape: (8878, 4)


,tweet_id,text,clean_text,company
0,1950778,"@Tesco No, not really. You must be breaking tr...",no not really you must be breaking trades desc...,tesco
1,2124899,"@AmazonHelp Un po’ tutti quelli in italiano, o...",un po tutti quelli in italiano ora molto mr ro...,amazonhelp
2,1225712,@AmazonHelp Why while purchasing in amazon web...,why while purchasing in amazon website it does...,amazonhelp
3,991319,@AmericanAir Flight was 16 June 2017- Miami in...,flight was june miami intl to lhr flight aa so...,americanair
4,1666114,@AmazonHelp Do you have a direct email I can f...,do you have a direct email i can forward the c...,amazonhelp


In [6]:
# Create a small balanced sample for prompting evaluation
sampled_parts = []

# Loop through each company label
for company in sorted(test_df["company"].unique()):
    company_rows = test_df[test_df["company"] == company]
    
    # Sample up to 10 examples per company
    sampled_company_rows = company_rows.sample(
        n=min(len(company_rows), 10),
        random_state=42
    )
    
    sampled_parts.append(sampled_company_rows)

# Combine all sampled rows into one dataframe
prompt_sample = pd.concat(sampled_parts, axis=0).reset_index(drop=True)

# Check the result
print("Prompt sample shape:", prompt_sample.shape)
print(prompt_sample["company"].value_counts())

Prompt sample shape: (90, 4)
company
amazonhelp      10
americanair     10
applesupport    10
delta           10
southwestair    10
spotifycares    10
tesco           10
uber_support    10
virgintrains    10
Name: count, dtype: int64


A smaller balanced sample is used for prompting evaluation.  
This keeps the experiment manageable while still including examples from every company class.

In [7]:
# company labels
labels = sorted(test_df["company"].unique())

print("Valid labels:")
print(labels)

Valid labels:
['amazonhelp', 'americanair', 'applesupport', 'delta', 'southwestair', 'spotifycares', 'tesco', 'uber_support', 'virgintrains']


In [8]:
# Function to create a zero-shot classification prompt
def create_prompt(tweet_text, labels):
    label_list = ", ".join(labels)
    
    prompt = f"""
You are classifying customer support tweets.

Choose the company label that best matches the tweet.

Valid labels:
{label_list}

Tweet:
{tweet_text}

Return only one label from the valid labels.
"""
    return prompt.strip()

In [11]:
# Test the prompt format on one example
example_text = prompt_sample.iloc[0]["clean_text"]
example_prompt = create_prompt(example_text, labels)

print(example_prompt)

You are classifying customer support tweets.

Choose the company label that best matches the tweet.

Valid labels:
amazonhelp, americanair, applesupport, delta, southwestair, spotifycares, tesco, uber_support, virgintrains

Tweet:
thank you its now up to the st november estimated

Return only one label from the valid labels.


In [13]:
# Use tweets per company from the balanced prompting sample
prompt_eval = prompt_sample.copy()

# Add ID for each prompt
prompt_eval["prompt_id"] = range(1, len(prompt_eval) + 1)

print("Prompting evaluation sample:", prompt_eval.shape)
print(prompt_eval["company"].value_counts())

Prompting evaluation sample: (90, 5)
company
amazonhelp      10
americanair     10
applesupport    10
delta           10
southwestair    10
spotifycares    10
tesco           10
uber_support    10
virgintrains    10
Name: count, dtype: int64


In [14]:
# Create prompts for each tweet
prompt_eval["prompt"] = prompt_eval["clean_text"].apply(
    lambda text: create_prompt(text, labels)
)

# Show the first prompt
print(prompt_eval.loc[0, "prompt"])

You are classifying customer support tweets.

Choose the company label that best matches the tweet.

Valid labels:
amazonhelp, americanair, applesupport, delta, southwestair, spotifycares, tesco, uber_support, virgintrains

Tweet:
thank you its now up to the st november estimated

Return only one label from the valid labels.


In [15]:
# Save prompts for manual ChatGPT classification
prompt_eval[["prompt_id", "prompt"]].to_csv(
    "../Data/manual_prompts.csv",
    index=False
)

# Save true labels separately for evaluation later
prompt_eval[["prompt_id", "clean_text", "company"]].to_csv(
    "../Data/manual_prompt_true_labels.csv",
    index=False
)

print("Manual prompting files saved.")

Manual prompting files saved.


## Evaluating the Prompting-Based Model

After manually collecting the LLM predictions, we compare them with the true company labels.  
We use the same metrics as the other classification models: accuracy, macro F1-score, weighted F1-score, and a classification report.

In [16]:
# Load true labels and manual LLM predictions
true_labels = pd.read_csv("../Data/manual_prompt_true_labels.csv")
predictions = pd.read_csv("../Data/manual_prompt_predictions.csv")

# Rename prediction column if it is called "label"
predictions = predictions.rename(columns={"label": "llm_prediction"})

# Preview both files
print("True labels shape:", true_labels.shape)
print("Predictions shape:", predictions.shape)

predictions.head()

True labels shape: (90, 3)
Predictions shape: (90, 2)


,prompt_id,llm_prediction
0,1,amazonhelp
1,2,amazonhelp
2,3,amazonhelp
3,4,amazonhelp
4,5,amazonhelp


In [17]:
# Merge true labels and predictions using prompt_id
prompt_results = true_labels.merge(
    predictions,
    on="prompt_id",
    how="inner"
)

# Check merged result
print("Merged results shape:", prompt_results.shape)
prompt_results.head()

Merged results shape: (90, 4)


,prompt_id,clean_text,company,llm_prediction
0,1,thank you its now up to the st november estimated,amazonhelp,amazonhelp
1,2,hi spoke to cs they said i have to return it m...,amazonhelp,amazonhelp
2,3,done filled the required info awaiting your ac...,amazonhelp,amazonhelp
3,4,the sale period is already over and i could ne...,amazonhelp,amazonhelp
4,5,when will be the earliest more month,amazonhelp,amazonhelp


In [18]:
# True labels
y_true = prompt_results["company"]

# LLM predictions
y_pred = prompt_results["llm_prediction"]

# Calculate scores
prompt_accuracy = accuracy_score(y_true, y_pred)
prompt_macro_f1 = f1_score(y_true, y_pred, average="macro")
prompt_weighted_f1 = f1_score(y_true, y_pred, average="weighted")

print("Prompting Model Accuracy:", round(prompt_accuracy, 4))
print("Prompting Model Macro F1:", round(prompt_macro_f1, 4))
print("Prompting Model Weighted F1:", round(prompt_weighted_f1, 4))

Prompting Model Accuracy: 0.9778
Prompting Model Macro F1: 0.9771
Prompting Model Weighted F1: 0.9771


In [19]:
# Detailed class-level performance
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

  amazonhelp       1.00      1.00      1.00        10
 americanair       0.91      1.00      0.95        10
applesupport       1.00      0.80      0.89        10
       delta       1.00      1.00      1.00        10
southwestair       1.00      1.00      1.00        10
spotifycares       0.91      1.00      0.95        10
       tesco       1.00      1.00      1.00        10
uber_support       1.00      1.00      1.00        10
virgintrains       1.00      1.00      1.00        10

    accuracy                           0.98        90
   macro avg       0.98      0.98      0.98        90
weighted avg       0.98      0.98      0.98        90



In [20]:
# Save final prompting model scores
prompting_results_df = pd.DataFrame([
    {
        "model": "Zero-shot prompting",
        "sample_size": len(prompt_results),
        "test_accuracy": prompt_accuracy,
        "test_macro_f1": prompt_macro_f1,
        "test_weighted_f1": prompt_weighted_f1
    }
])

# Save table for report
prompting_results_df.to_csv(
    "../results/tables/prompting_model_results.csv",
    index=False
)

prompting_results_df

,model,sample_size,test_accuracy,test_macro_f1,test_weighted_f1
0,Zero-shot prompting,90,0.977778,0.977072,0.977072


## Prompting Model Result 
The zero-shot prompting model performed very well on the balanced sample of 90 tweets.  
It achieved an accuracy of about 0.98 and a macro F1-score of about 0.98.

This shows that the LLM was able to correctly match most customer tweets to the right company label.  
However, this result should be treated with some caution. The prompting model was tested on a much smaller sample than the classical and neural models, so the results are not fully comparable.

Overall, the prompting approach looks promising, but a larger evaluation sample would be needed to make a stronger conclusion.

In [21]:
# Show incorrect prompting predictions
prompt_errors = prompt_results[prompt_results["company"] != prompt_results["llm_prediction"]]

print("Number of errors:", len(prompt_errors))
prompt_errors

Number of errors: 2


,prompt_id,clean_text,company,llm_prediction
20,21,id like to offer thanks to for their assistanc...,applesupport,americanair
22,23,and this spotify is streaming and can control ...,applesupport,spotifycares


In [ ]:
# Save prompting predictions for later analysis
prompt_results.to_csv(
    "../results/tables/prompting_detailed_results.csv",
    index=False
)

print("Detailed prompting results saved.")

Detailed prompting results saved.
